In [6]:
import os
import requests
from urllib.request import urlretrieve
import pandas as pd
import json

BUNGIE_URL = "https://bungie.net"
MANIFEST_PATH = "/Platform/Destiny2/Manifest/"

AVAILABLE_LANGS = ["en", "fr"]
TABLES = ["DestinyInventoryItemDefinition", "DestinyDamageTypeDefinition", "DestinyItemTierTypeDefinition","DestinyItemCategoryDefinition"]

API_DATA_FOLDER = "api_data"
REFS_FOLDER = "api_data\REFS" # type: ignore


TYPE_CATEGORY_HASHES = json.load(open(os.path.join(REFS_FOLDER,"type_category_hashes.json"), "r"))
TYPE_HASHES = json.load(open(os.path.join(REFS_FOLDER,"type_hashes.json"), "r"))
CATEGORY_HASHES = json.load(open(os.path.join(REFS_FOLDER,"category_hashes.json"), "r"))
OBJECT_TYPE_HASHES = json.load(open(os.path.join(REFS_FOLDER,"object_type_hashes.json"), "r"))
TIER_HASHES = json.load(open(os.path.join(REFS_FOLDER, "tier_hashes.json"), "r"))
DAMAGE_TYPE_HASHES = json.load(open(os.path.join(REFS_FOLDER, "damage_type_hashes.json"), "r"))

for lang in AVAILABLE_LANGS:
    lang_path = os.path.join(API_DATA_FOLDER, lang)
    if not os.path.exists(lang_path):
        os.makedirs(lang_path)


## Bungie API Functions

def save_manifest_version(version):
    with open(os.path.join(API_DATA_FOLDER,"manifest_version.txt"), "w+") as f:
        f.write(version)
        f.close()



def check_manifest_update():
    def get_current_version():
        with open(os.path.join(API_DATA_FOLDER,"manifest_version.txt"), "r") as f:
            version = f.readline()
        return version
    
    version = get_current_version()
    manifest_url = BUNGIE_URL + MANIFEST_PATH
    try :
        manifest = requests.get(manifest_url).json()
        manifest_version = manifest.get("Response").get('version', '')
        if manifest_version == '':
            raise Exception("Error when reading manifest")
        return manifest_version == version
    except Exception as e:
        print("Error when getting manifest version : ", e)

def get_world_content_data_json(lang):
    manifest_url = BUNGIE_URL + MANIFEST_PATH
    try :
        if lang not in AVAILABLE_LANGS:
            raise Exception(f"Language '{lang}' is not in available languages")
        manifest = requests.get(manifest_url).json()
        manifest_content_url = manifest.get('Response').get("jsonWorldComponentContentPaths").get(lang)
        for table in TABLES:
            filename = os.path.join(API_DATA_FOLDER, lang, f"{table}_{lang}.json")
            table_url = BUNGIE_URL + manifest_content_url.get(table)
            urlretrieve(table_url, filename)
    except Exception as e:
        print("Error when getting world content :", e)



## Database functions

def _filter_type_category(entry, type_category):
    hashes = TYPE_CATEGORY_HASHES[type_category]
    try:
        if entry["itemCategoryHashes"][hashes[0]] == hashes[1]:
            return True
        else : 
            return False
    except:
        return False

def _get_value(entry, key):
    return entry.get(key)

def _get_index(entry, index):
    return entry[index]

def _get_type(itemCategoryHashes):
    return list(set(itemCategoryHashes) & set(TYPE_HASHES.values()))[-1]

def _get_category(itemCategoryHashes):
    return list(set(itemCategoryHashes) & set(CATEGORY_HASHES.values()))[0]

def _filter_tier(entry, tier):
    hash = TIER_HASHES[tier]
    if entry["inventory"]["tierTypeHash"] == hash:
        return True
    else : 
        return False
    


def get_tiers():
    tiers_dfs = {}
    for lang in AVAILABLE_LANGS:
        tiers_dfs[lang] = pd.read_json(os.path.join(API_DATA_FOLDER, lang, f"DestinyItemTierTypeDefinition_{lang}.json")).T[["displayProperties", "hash"]]
        tiers_dfs[lang]["displayProperties"] = tiers_dfs[lang]["displayProperties"].apply(_get_value, args=("name",))
        
        tiers_dfs[lang].rename(columns = {"displayProperties" : f"tier_{lang}"}, inplace=True)
    
    df = tiers_dfs[AVAILABLE_LANGS[0]]
    for i in range(1,len(AVAILABLE_LANGS)):
        df = df.merge(tiers_dfs[AVAILABLE_LANGS[i]],how="left", on="hash")
    
    return df

def get_damage_types():
    dt_dfs = {}
    for lang in AVAILABLE_LANGS:
        dt_dfs[lang] = pd.read_json(os.path.join(API_DATA_FOLDER, lang, f"DestinyDamageTypeDefinition_{lang}.json")).T[["displayProperties",  "hash"]]
        
        dt_dfs[lang]["iconLink"] = dt_dfs[lang]["displayProperties"].apply(_get_value, args=("icon",))
        dt_dfs[lang]["displayProperties"] = dt_dfs[lang]["displayProperties"].apply(_get_value, args=("name",))

        dt_dfs[lang].rename(columns = {"displayProperties" : f"damageType_{lang}"}, inplace=True)
    
    df = dt_dfs[AVAILABLE_LANGS[0]]
    for i in range(1,len(AVAILABLE_LANGS)):
        df = df.merge(dt_dfs[AVAILABLE_LANGS[i]],how="left", on=["hash", "iconLink"])


    return df[df["hash"].isin(list(DAMAGE_TYPE_HASHES.values()))]


def get_types():
    dt_dfs = {}
    for lang in AVAILABLE_LANGS:
        dt_dfs[lang] = pd.read_json(os.path.join(API_DATA_FOLDER, lang, f"DestinyItemCategoryDefinition_{lang}.json")).T[["displayProperties",  "hash"]]
        dt_dfs[lang]["displayProperties"] = dt_dfs[lang]["displayProperties"].apply(_get_value, args=("name",))

        dt_dfs[lang].rename(columns = {"displayProperties" : f"type_{lang}"}, inplace=True)
    
    df = dt_dfs[AVAILABLE_LANGS[0]]
    for i in range(1,len(AVAILABLE_LANGS)):
        df = df.merge(dt_dfs[AVAILABLE_LANGS[i]],how="left", on="hash")
    

    return df[df["hash"].isin(TYPE_HASHES.values())]

def get_categories():
    dt_dfs = {}
    for lang in AVAILABLE_LANGS:
        dt_dfs[lang] = pd.read_json(os.path.join(API_DATA_FOLDER, lang, f"DestinyItemCategoryDefinition_{lang}.json")).T[["displayProperties",  "hash"]]
        dt_dfs[lang]["displayProperties"] = dt_dfs[lang]["displayProperties"].apply(_get_value, args=("name",))

        dt_dfs[lang].rename(columns = {"displayProperties" : f"category_{lang}"}, inplace=True)
    
    df = dt_dfs[AVAILABLE_LANGS[0]]
    for i in range(1,len(AVAILABLE_LANGS)):
        df = df.merge(dt_dfs[AVAILABLE_LANGS[i]],how="left", on="hash")
    

    return df[df["hash"].isin(CATEGORY_HASHES.values())]


def get_weapons():
    weapons_dfs = {}
    for lang in AVAILABLE_LANGS:
        weapons_dfs[lang] = pd.read_json(os.path.join(API_DATA_FOLDER, lang, f"DestinyInventoryItemDefinition_{lang}.json")).T[["displayProperties","hash", "damageTypeHashes", "screenshot", "flavorText", "inventory", "itemCategoryHashes", "defaultDamageTypeHash", "collectibleHash", "tooltipNotifications"]]
        weapons_dfs[lang] = weapons_dfs[lang][weapons_dfs[lang]["hash"] != 2251716886]
        weapons_dfs[lang] = weapons_dfs[lang][weapons_dfs[lang].apply(_filter_tier, args=("Exotic",), axis=1)]
        weapons_dfs[lang] = weapons_dfs[lang][weapons_dfs[lang].apply(_filter_type_category, args=("Weapon",), axis=1)]
        weapons_dfs[lang]["inventory"] = weapons_dfs[lang]["inventory"].apply(_get_value, args=("tierTypeHash",))
        weapons_dfs[lang]["categoryHash"] = weapons_dfs[lang]["itemCategoryHashes"].apply(_get_category)
        weapons_dfs[lang]["itemCategoryHashes"] = weapons_dfs[lang]["itemCategoryHashes"].apply(_get_type)
        weapons_dfs[lang]["iconLink"] = weapons_dfs[lang]["displayProperties"].apply(_get_value, args=("icon",))
        weapons_dfs[lang]["displayProperties"] = weapons_dfs[lang]["displayProperties"].apply(_get_value, args=("name",))
        weapons_dfs[lang]["tooltipNotifications"] = weapons_dfs[lang]["tooltipNotifications"].apply(len) if "tooltipNotifications" in weapons_dfs[lang] else 0

        weapons_dfs[lang].rename(columns = {"displayProperties" : f"weaponName_{lang}", "flavorText" : f"flavorText_{lang}", "inventory" : "tierTypeHash", "screenshot" : "screenshotLink", "itemCategoryHashes" : "typeHash"}, inplace=True)
        

    
    df = weapons_dfs[AVAILABLE_LANGS[0]]
    for i in range(1,len(AVAILABLE_LANGS)):
        df = df.merge(weapons_dfs[AVAILABLE_LANGS[i]],how="left", on=["hash", "tierTypeHash", "typeHash", "categoryHash", "iconLink", "screenshotLink", "defaultDamageTypeHash", "collectibleHash", "tooltipNotifications"])

    df.dropna(subset=["damageTypeHashes_x", "collectibleHash"], inplace=True)
    df = df.sort_values(by="tooltipNotifications", ascending=True).drop_duplicates(subset=["weaponName_en"], keep="first")
    return df

In [7]:
get_damage_types()

,damageType_en,hash,iconLink,damageType_fr
0,Kinetic,3373582085,/common/destiny2_content/icons/DestinyDamageTy...,Cinétiques
1,Arc,2303181850,/common/destiny2_content/icons/DestinyDamageTy...,Cryo-électriques
2,Solar,1847026933,/common/destiny2_content/icons/DestinyDamageTy...,Solaires
3,Void,3454344768,/common/destiny2_content/icons/DestinyDamageTy...,Abyssaux
5,Stasis,151347233,/common/destiny2_content/icons/DestinyDamageTy...,Stase
6,Strand,3949783978,/common/destiny2_content/icons/DestinyDamageTy...,Filobscur


In [8]:
get_types()

,type_en,hash,type_fr
4,Auto Rifle,5,Fusils automatiques
5,Hand Cannon,6,Revolvers
6,Pulse Rifle,7,Fusils à impulsion
7,Scout Rifle,8,Fusils d'éclaireur
8,Fusion Rifle,9,Fusils à fusion
9,Sniper Rifle,10,Fusils de précision
10,Shotgun,11,Fusils à pompe
11,Machine Gun,12,Mitrailleuses
12,Rocket Launcher,13,Lance-roquettes
13,Sidearm,14,Pistolets


In [9]:
get_categories()

,category_en,hash,category_fr
1,Kinetic Weapon,2,Arme cinétique
2,Energy Weapon,3,Arme énergétique
3,Power Weapon,4,Arme puissante


In [10]:
info = get_weapons()

In [11]:
info

,weaponName_en,hash,damageTypeHashes_x,screenshotLink,flavorText_en,tierTypeHash,typeHash,defaultDamageTypeHash,collectibleHash,tooltipNotifications,categoryHash,iconLink,weaponName_fr,damageTypeHashes_y,flavorText_fr
0,Coldheart,1345867571,[2303181850],/common/destiny2_content/screenshots/134586757...,The latest Omolon engineering leverages liquid...,2759499571,2489664120,2303181850,1657028070,0,3,/common/destiny2_content/icons/2a2a95b819312fc...,Cœur de glace,[2303181850],Ce tout nouvel équipement signé Omolon utilise...
103,Lorentz Driver,3761898871,[3454344768],/common/destiny2_content/screenshots/376189887...,"""Weapon system no longer explodes when trigger...",2759499571,9,3454344768,3602500492,0,3,/common/destiny2_content/icons/4d6d145ff3a33a0...,Impulsion Lorentz,[3454344768],« Le système d'arme n'explose plus quand on ap...
102,Vex Mythoclast,4289226715,[1847026933],/common/destiny2_content/screenshots/428922671...,"…a causal loop within the weapon's mechanism, ...",2759499571,9,1847026933,2300465938,0,3,/common/destiny2_content/icons/111a10b59029fc6...,Exégèse du Vex,[1847026933],...une boucle de causalité à l'intérieur du mé...
101,Cryosthesia 77K,603721696,[151347233],/common/destiny2_content/screenshots/603721696...,There are things colder than cold.,2759499571,14,151347233,2817803243,0,2,/common/destiny2_content/icons/285bb2a7a6f016c...,Cryosthésie 77K,[151347233],Certaines choses sont plus froides que le froid.
99,Dead Man's Tale,3654674561,[3373582085],/common/destiny2_content/screenshots/365467456...,"""Long, short, they all end the same way."" —Kat...",2759499571,8,3373582085,3723101298,0,2,/common/destiny2_content/icons/cfc2c246cfd404d...,Récit d'un homme mort,[3373582085],"« Longs ou courts, ils finissent tous de la mê..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50,Ace of Spades,347366834,[3373582085],/common/destiny2_content/screenshots/347366834...,"""Folding was never an option."" —Cayde-6",2759499571,6,3373582085,1660030046,0,2,/common/destiny2_content/icons/cdfbfd3f098329a...,As de pique,[3373582085],« Se coucher n'a jamais été une option. » – Ca...
47,Two-Tailed Fox,2694576561,[3454344768],/common/destiny2_content/screenshots/269457656...,Adorably murderous.,2759499571,13,3454344768,199171384,0,4,/common/destiny2_content/icons/c065e1c0d03d964...,Renard à deux queues,[3454344768],Adorablement meurtrier.
125,Vexcalibur,3118061005,[3454344768],/common/destiny2_content/screenshots/311806100...,"And they came to a lake fair and broad. ""Lo,"" ...",2759499571,3871742104,3454344768,2689028695,3,3,/common/destiny2_content/icons/735baec8c23ff14...,Vexcalibur,[3454344768],Et ils arrivèrent à un lac aussi beau que gran...
136,Wish-Keeper,2910326942,[3949783978],/common/destiny2_content/screenshots/291032694...,"""Live."" —Taranis Rivensmate",2759499571,3317538576,3949783978,221021254,3,2,/common/destiny2_content/icons/6b47d872840188b...,Garde-souhaits,[3949783978],"« Vivez. » – Taranis, Compagnon de Riven"


In [3]:
import os
import requests
from urllib.request import urlretrieve
import pandas as pd
import json

BUNGIE_URL = "https://bungie.net"
MANIFEST_PATH = "/Platform/Destiny2/Manifest/"

AVAILABLE_LANGS = ["en", "fr"]
TABLES = ["DestinyInventoryItemDefinition", "DestinyDamageTypeDefinition", "DestinyItemTierTypeDefinition","DestinyItemCategoryDefinition"]

API_DATA_FOLDER = "api_data"
REFS_FOLDER = "api_data\REFS" # type: ignore

In [4]:

CORRECTION_TABLE = json.load(open(os.path.join( REFS_FOLDER,"correction_table.json"), "r"))

In [5]:
CORRECTION_TABLE

{'TypeDefinition': {'type_en': {'Trace Rifles': 'Trace Rifle',
   'Grenade Launchers': 'Grenade Launcher',
   'Submachine Guns': 'Submachine Gun',
   'Linear Fusion Rifles': 'Linear Fusion Rifle',
   'Glaives': 'Glaive',
   'Bows': 'Bow'},
  'type_fr': {'Fusils automatiques': 'Fusil automatique',
   'Revolvers': 'Revolver',
   'Fusils a impulsion': 'Fusil à impulsion',
   'Fusils à éclaireur': 'Fusil à éclaireur',
   'Fusils à fusion': 'Fusil à fusion',
   'Mitrailleuses': 'Mitrailleuse',
   'Pistolets': 'Pistolet',
   'Épées': 'Épée',
   'Fusils à rayon': 'Fusil à rayon',
   'Pistolets-mitrailleurs': 'Pistolet-mitrailleur',
   'Fusils à fusion linéaire': 'Fusil à fusion linéaire',
   'Glaives': 'Glaive',
   'Arcs': 'Arc'}},
 'DamageTypeDefinition': {'damageType_fr': {'Cinétiques': 'Cinétique',
   'Cryo-électriques': 'Cryo-électrique',
   'Solaires': 'Solaire',
   'Abyssaux': 'Abyssal'}}}